<a href="https://colab.research.google.com/github/xyma8/ML-kurs/blob/main/kurs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Подготовка данных. Приведение к сущностям (srv_res)

## Подключение google drive и создание dataframe (DF) списка всех студентов

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
studentsDF = pd.read_csv("/content/gdrive/MyDrive/kurs/csv_res/students.csv")
print(studentsDF)

## Функция чтения исходных csv с оценками за аттестацию и создание DF с студентами по id и их оценками (по всем группам)

In [ ]:
import pandas as pd


def build_grades_attest_table(
    grades_path: str,
    studentsDF: pd.DataFrame,
    group,
    specialization,
    course,
    semester,
    output_path: str = "result_grades.csv",
):
    """
    grades_path: путь к таблице оценок

    studentsDF: уже загруженный DataFrame студентов вида:
        id, full_name, group, specialization, course

    group, specialization, course: значения для фильтрации студентов

    semester: номер семестра, который вручную добавится в итоговую таблицу

    output_path: путь для сохранения результата
    """

    # Читаем таблицу оценок без заголовков
    raw = pd.read_csv(grades_path, header=None)

    # Первая строка содержит названия предметов, начиная с 3-го столбца
    subjects = raw.iloc[0, 2:].tolist()

    # Данные студентов начинаются с 3-й строки
    grades = raw.iloc[2:].copy()

    # Второй столбец — ФИО
    grades = grades.rename(columns={1: "full_name"})

    # Оставляем ФИО + оценки по предметам
    subject_columns = list(range(2, raw.shape[1]))
    grades = grades[["full_name"] + subject_columns]

    # Переименовываем столбцы оценок в названия предметов
    rename_map = {
        col: subject
        for col, subject in zip(subject_columns, subjects)
    }
    grades = grades.rename(columns=rename_map)

    # Убираем строки без ФИО
    grades = grades.dropna(subset=["full_name"])

    # Чистим ФИО
    grades["full_name"] = grades["full_name"].astype(str).str.strip()

    # Переводим таблицу оценок в длинный формат:
    # full_name | subject | grade
    grades_long = grades.melt(
        id_vars=["full_name"],
        var_name="subject",
        value_name="grade"
    )

    # Убираем пустые оценки
    grades_long = grades_long.dropna(subset=["grade"])

    # Работаем с копией studentsDF, чтобы не менять исходный DataFrame
    students = studentsDF.copy()

    # Чистим ФИО
    students["full_name"] = students["full_name"].astype(str).str.strip()

    # Фильтруем студентов
    students_filtered = students[
        (students["group"].astype(str) == str(group)) &
        (students["specialization"].astype(str) == str(specialization)) &
        (students["course"].astype(str) == str(course))
    ].copy()

    # Соединяем оценки со студентами по ФИО
    result = grades_long.merge(
        students_filtered[["id", "full_name"]],
        on="full_name",
        how="inner"
    )

    # Переименовываем id в student_id
    result = result.rename(columns={"id": "student_id"})

    # Добавляем semester
    result["semester"] = semester

    # Итоговый порядок столбцов
    result = result[["student_id", "subject", "grade", "semester"]]

    # Сохраняем результат
    result.to_csv(output_path, index=False, encoding="utf-8-sig")

    return result

In [ ]:
grades_attest_24isbo1 = build_grades_attest_table(
    grades_path="/content/gdrive/MyDrive/kurs/attest_24isbo1_sem1.csv",
    studentsDF=studentsDF,
    group=1,
    specialization="ИСбо",
    course=24,
    semester=1,
    output_path="/content/gdrive/MyDrive/kurs/csv_res/attest_24isbo1_sem1.csv"
)

In [ ]:
grades_attest_24isbo2 = build_grades_attest_table(
    grades_path="/content/gdrive/MyDrive/kurs/attest_24isbo2_sem1.csv",
    studentsDF=studentsDF,
    group=2,
    specialization="ИСбо",
    course=24,
    semester=1,
    output_path="/content/gdrive/MyDrive/kurs/csv_res/attest_24isbo2_sem1.csv"
)

In [ ]:
grades_attest_24isbo3 = build_grades_attest_table(
    grades_path="/content/gdrive/MyDrive/kurs/attest_24isbo3_sem1.csv",
    studentsDF=studentsDF,
    group=3,
    specialization="ИСбо",
    course=24,
    semester=1,
    output_path="/content/gdrive/MyDrive/kurs/csv_res/attest_24isbo3_sem1.csv"
)

In [ ]:
grades_attest_24isbo4 = build_grades_attest_table(
    grades_path="/content/gdrive/MyDrive/kurs/attest_24isbo4_sem1.csv",
    studentsDF=studentsDF,
    group=4,
    specialization="ИСбо",
    course=24,
    semester=1,
    output_path="/content/gdrive/MyDrive/kurs/csv_res/attest_24isbo4_sem1.csv"
)